In [0]:
%run "../../utils"

In [0]:

# Load source tables from the silver layer
v = spark.table(f"{silver_schema}.epa_vehicles").alias("v")
e = spark.table(f"{silver_schema}.epa_emissions").alias("e")

# Join vehicles and emissions, group by year and make, and aggregate smartway score and record count
df = (
    v.join(e, col("v.vehicle_id") == col("e.vehicle_id"), "inner")
     .groupBy(col("v.year").alias("year"), col("v.make").alias("make"))
     .agg(
         avg(col("e.smartway_score")).alias("avg_smartway_score"),
         spark_count("*").alias("records")
     )
     .orderBy(col("year").desc(), col("avg_smartway_score").desc())
)

# Write the aggregated results to the gold layer as a Delta table
df.write.format("delta").mode("overwrite").saveAsTable(f"{gold_schema}.epa_smartway_by_make_year")